# Análisis ofensivo de MLB — Temporada regular 2025

## Objetivo
Comparar el rendimiento de bateadores considerando sus oportunidades al bate.



In [ ]:
import requests

In [ ]:
url = "https://statsapi.mlb.com/api/v1/teams"
parametros = {"sportId": 1, "season": 2025}

respuesta = requests.get(url, params=parametros, timeout=30)

print(respuesta.status_code)
respuesta.raise_for_status()

In [ ]:
datos = respuesta.json()
print(datos.keys())

In [ ]:
equipos = datos["teams"]
print(type(equipos))
print(len(equipos))

In [ ]:
equipos[0]

In [ ]:
equipos[0]["name"]

In [ ]:
equipos[0]["id"]

In [ ]:
for equipo in equipos:
    print(equipo["id"], equipo["name"])

In [ ]:
equipo = equipos[0]

fila = {
    "equipo_id": equipo["id"],
    "nombre": equipo["name"],
    "liga": equipo["league"]["name"],
    "division": equipo["division"]["name"],
    "temporada": equipo["season"]
}

fila

In [ ]:
filas_equipos = []

for equipo in equipos:
    fila = {
        "equipo_id": equipo["id"],
        "nombre": equipo["name"],
        "liga": equipo["league"]["name"],
        "division": equipo["division"]["name"],
        "temporada": equipo["season"]
    }
    filas_equipos.append(fila)

print(len(filas_equipos))

In [ ]:
import pandas as pd

df_equipos = pd.DataFrame(filas_equipos)

df_equipos.head()

In [ ]:
print("Filas y columnas:", df_equipos.shape)

print("\nDatos faltantes por columna:")
print(df_equipos.isna().sum())

print("\nEquipos repetidos en la misma temporada:")
print(df_equipos.duplicated(
    subset=["equipo_id", "temporada"]
).sum())

In [ ]:
from pathlib import Path

carpeta = Path("../data/processed")
carpeta.mkdir(parents=True, exist_ok=True)

archivo = carpeta / "equipos_2025.csv"
df_equipos.to_csv(archivo, index=False)

print(archivo.resolve())

In [ ]:
url_stats = "https://statsapi.mlb.com/api/v1/stats"

parametros_stats = {
    "stats": "season",
    "group": "hitting",
    "season": 2025,
    "sportIds": 1,
    "gameType": "R",
    "playerPool": "ALL",
    "limit": 1
}

respuesta_stats = requests.get(
    url_stats,
    params=parametros_stats,
    timeout=30
)

print(respuesta_stats.status_code)

In [ ]:
respuesta_stats.raise_for_status()
datos_stats = respuesta_stats.json()

print(datos_stats.keys())

In [ ]:
estadisticas = datos_stats["stats"]
print(type(estadisticas))
print(len(estadisticas))

In [ ]:
bloque = estadisticas[0]
print(bloque.keys())

In [ ]:
registros = bloque["splits"]
print(len(registros))

registro = registros[0]
print(registro.keys())

In [ ]:
print(registro["player"])
print(registro["stat"])

In [ ]:
fila_bateador = {
    "jugador_id": registro["player"]["id"],
    "nombre": registro["player"]["fullName"],
    "temporada": int(registro["season"]),
    "hits": registro["stat"]["hits"],
    "turnos": registro["stat"]["atBats"],
    "home_runs": registro["stat"]["homeRuns"],
    "apariciones_plato": registro["stat"]["plateAppearances"]
}

fila_bateador

In [ ]:
print("Registros disponibles:", bloque["totalSplits"])
print("Registros recibidos:", len(registros))

In [ ]:
parametros_stats["limit"] = bloque["totalSplits"]

respuesta_stats = requests.get(
    url_stats,
    params=parametros_stats,
    timeout=30
)
respuesta_stats.raise_for_status()

datos_stats = respuesta_stats.json()
bloque = datos_stats["stats"][0]
registros = bloque["splits"]

print("Disponibles:", bloque["totalSplits"])
print("Recibidos:", len(registros))
assert len(registros) == bloque["totalSplits"], "Respuesta incompleta: revisar paginación de la API."

In [ ]:
ids_jugadores = []

for registro in registros:
    ids_jugadores.append(registro["player"]["id"])

print("Registros:", len(ids_jugadores))
print("Jugadores únicos:", len(set(ids_jugadores)))

In [ ]:
filas_bateadores = []

for registro in registros:
    fila = {
        "jugador_id": registro["player"]["id"],
        "nombre": registro["player"]["fullName"],
        "temporada": int(registro["season"]),
        "hits": registro["stat"]["hits"],
        "turnos": registro["stat"]["atBats"],
        "home_runs": registro["stat"]["homeRuns"],
        "apariciones_plato": registro["stat"]["plateAppearances"],
        "numero_equipos": registro["numTeams"]
    }
    filas_bateadores.append(fila)

df_bateadores = pd.DataFrame(filas_bateadores)
df_bateadores.head()

In [ ]:
varios_equipos = df_bateadores[
    df_bateadores["numero_equipos"] > 1
]

print("Jugadores con varios equipos:", len(varios_equipos))

varios_equipos[["nombre", "numero_equipos"]].head()

In [ ]:
print("Filas y columnas:", df_bateadores.shape)
print("\nDatos faltantes:")
print(df_bateadores.isna().sum())

print("\nJugador-temporada repetidos:")
print(df_bateadores.duplicated(
    subset=["jugador_id", "temporada"]
).sum())

In [ ]:
inconsistencias = df_bateadores[
    (df_bateadores["hits"] > df_bateadores["turnos"]) |
    (df_bateadores["home_runs"] > df_bateadores["hits"]) |
    (df_bateadores["turnos"] > df_bateadores["apariciones_plato"])
]

print("Registros inconsistentes:", len(inconsistencias))

In [ ]:
archivo_bateadores = carpeta / "bateadores_2025.csv"

df_bateadores.to_csv(archivo_bateadores, index=False)

print(archivo_bateadores.resolve())

In [ ]:
archivo_json = carpeta / "bateadores_2025.json"

df_bateadores.to_json(
    archivo_json,
    orient="records",
    force_ascii=True,
    indent=2
)

print(archivo_json.resolve())

In [ ]:
df_bateadores.nlargest(5, "hits")[["nombre", "hits"]]

In [ ]:
print(respuesta_stats.url)

In [ ]:
df_bateadores.nlargest(5,"home_runs")[["nombre","home_runs"]]